# I-RAVEN in PyNeuraLogic — minimal port of the best GNN config

A deliberately small transcription of the winning torch model
(`gnn` = `RelGNLayer`, `object_only`, per-candidate scoring) into a PyNeuraLogic template.

**Status — read before you trust the numbers.** This runs end to end and learns something
(~20% val on `center_single` vs 12.5% chance), but it does **not** reach the torch baseline
(75.4% on `center_single`, 68.8% overall on I-RAVEN). The message-passing layers still collapse
to a constant output after a few epochs. See the last cell for the diagnosis and what to try next.

**Setup:** `conda activate pynl` and `ml load Java/21`. The kernel must be the `pynl` env
(Python 3.11, neuralogic 0.9.2) — PyNeuraLogic runs its engine on the JVM through JPype.

In [1]:
import glob, os, xml.etree.ElementTree as ET
import numpy as np
from neuralogic.core import Model, R, V, Settings, Transformation, Aggregation, Metadata, Combination, Combination
from neuralogic.dataset import Dataset, Sample
from neuralogic.nn import Adam, CrossEntropy
import neuralogic

neuralogic.set_max_memory_size(16)   # JVM heap; the default is small enough to kill a kernel

print("neuralogic", neuralogic.__version__, "| JAVA_HOME", os.environ.get("JAVA_HOME", "<unset>"))

neuralogic 0.9.2 | JAVA_HOME /mnt/appl/software/Java/21.0.8


/home/kassyask/.conda/envs/pynl/lib/python3.11/site-packages/neuralogic/core/builder/builder.py:4: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


## 1. Config

In [2]:
DATA          = "data/I-RAVEN-10000"
CONSTELLATION = "center_single"   # simplest: 1 object per panel, no intra-panel spatial edges
N_TRAIN, N_VAL = 200, 80          # puzzles; each becomes 8 samples (one per candidate)
DIM, LAYERS, LR, EPOCHS = 16, 2, 0.005, 20

# constellation-local object slot -> universal position id (matches gnn/config.py CONFIG_POS_MAP)
POS_MAP = {
    "center_single": {0: 0},
    "distribute_four": {0: 1, 1: 2, 2: 3, 3: 4},
    "distribute_nine": {i: 5 + i for i in range(9)},
    "left_center_single_right_center_single": {0: 14, 1: 15},
    "up_center_single_down_center_single": {0: 16, 1: 17},
    "in_center_single_out_center_single": {0: 0, 1: 9},
    "in_distribute_four_out_center_single": {0: 0, 1: 18, 2: 19, 3: 20, 4: 21},
}
N_POS, N_COL, N_SHP, N_SIZ, N_PNL = 22, 10, 5, 6, 9

## 2. Read the ground truth out of the XML

Same parse as `gnn/data.py`: 16 panels (8 context, 8 candidates), each a list of objects with
position / type / size / colour, plus the bbox centre used to derive spatial relations.

In [3]:
def parse(xml_path, config):
    pos_map = POS_MAP[config]
    panels = []
    for panel in ET.parse(xml_path).getroot()[0]:
        objs, off = [], 0
        for struct in panel:
            for comp in struct:
                for layout in comp:
                    all_pos = eval(layout.attrib["Position"])
                    for e in layout:
                        b = eval(e.attrib["bbox"])
                        objs.append({
                            "pos":   pos_map[all_pos.index(b) + off],
                            "type":  int(e.attrib["Type"]) - 1,   # XML is 1-indexed
                            "size":  int(e.attrib["Size"]),
                            "color": int(e.attrib["Color"]),
                            "cx": b[1] + 0.5 * b[3], "cy": b[0] + 0.5 * b[2],
                        })
                    off += len(all_pos)
        panels.append(objs)
    return panels


def spatial_rels(a, b, eps=1e-4):
    r = []
    if   a["cx"] < b["cx"] - eps: r.append("left_of")
    elif a["cx"] > b["cx"] + eps: r.append("right_of")
    if   a["cy"] < b["cy"] - eps: r.append("above")
    elif a["cy"] > b["cy"] + eps: r.append("below")
    if abs(a["cy"] - b["cy"]) <= eps: r.append("same_row")
    if abs(a["cx"] - b["cx"]) <= eps: r.append("same_col")
    return r

RELATIONS = ["left_of", "right_of", "above", "below", "same_row", "same_col",
             "same_pos_cross_panel"]

## 3. One puzzle + one candidate -> one PyNeuraLogic sample

`object_only`: objects are the only nodes, no panel super-nodes. The edge that carries the
row/column rule is `same_pos_cross_panel` — it links objects occupying the same universal
position across panels. Dropping panel super-nodes was worth **+34 points** in the torch runs.

Two things that are easy to get wrong and cost a lot of debugging:

* **Every sample needs a unique query atom.** `Sample(R.score[label], ...)` with a nullary
  `score` makes all 8 candidates share one query, and PyNeuraLogic merges them — you silently
  get fewer samples than you passed in. Keying on a per-sample constant (`R.score(gid)`) fixes it.
* **The panel-identity fact is load-bearing.** Without `R.panel(...)` every panel is
  interchangeable, mean-pooling is fully symmetric, and the model cannot tell a row apart.

In [4]:
def sample_for(panels, cand, label, gid):
    keep = list(range(8)) + [8 + cand]           # 8 context panels + this one candidate
    facts, nodes = [R.graph(gid)], {}

    for p in keep:
        grid = p if p < 8 else 8                 # panel identity: 0-7 context, 8 = answer slot
        for o in panels[p]:
            nid = f"o{p}_{o['pos']}"
            nodes[nid] = (p, o)
            facts += [
                R.pos(nid,   f"p{o['pos']}"),   R.color(nid, f"c{o['color']}"),
                R.shape(nid, f"s{o['type']}"),  R.size(nid,  f"z{o['size']}"),
                R.panel(nid, f"g{grid}"),
                R.is_candidate(nid) if p == 8 + cand else R.is_context(nid),
            ]

    for p in keep:                               # intra-panel spatial edges
        ids = [n for n, (q, _) in nodes.items() if q == p]
        for i in ids:
            for j in ids:
                if i != j:
                    facts += [R.get(rel)(i, j) for rel in spatial_rels(nodes[i][1], nodes[j][1])]

    by_pos = {}                                  # cross-panel edges at matching position
    for n, (_, o) in nodes.items():
        by_pos.setdefault(o["pos"], []).append(n)
    for group in by_pos.values():
        for i in group:
            for j in group:
                if i != j:
                    facts.append(R.same_pos_cross_panel(i, j))

    return Sample(R.score(gid)[label], facts)


def load(split, n):
    files = sorted(glob.glob(os.path.join(DATA, CONSTELLATION, f"*_{split}.npz")))[:n]
    samples, groups = [], []
    for k, f in enumerate(files):
        panels = parse(f.replace(".npz", ".xml"), CONSTELLATION)
        target = int(np.load(f)["target"])
        groups.append((list(range(len(samples), len(samples) + 8)), target))
        samples += [sample_for(panels, c, 1 if c == target else 0, f"{split}{k}_{c}")
                    for c in range(8)]
    return samples, groups

## 4. The template

Line-for-line against `gnn/model.py`:

| torch | here |
|---|---|
| `NodeEncoder` — sum of `nn.Embedding` lookups | five rules sharing head `h0`, summed by default |
| `RelGNLayer` — `W([x_src + rel; x_dst; rel])` | one weighted rule per relation, body carries **both** `h(Src)` and `h(Dst)` |
| `scatter_mean` per destination | `Metadata(aggregation=Aggregation.AVG)` |
| residual `x + ...` | extra `h_k(X) :- h_{k-1}(X)` rule |
| `LayerNorm` + `ReLU` | `Transformation.LEAKY_RELU` (no LayerNorm equivalent — see notes) |
| ctx / cand mean-pool + `score_head(cat(...))` | two AVG rules + `{1,2D} score` with `Combination.CONCAT` |

Embedding tables are **weighted facts** (`R.emb_col("c3")[DIM,]`), one learnable D-vector per
attribute value. If you write `R.emb_col(V.C)[DIM,]` in a rule body without declaring those facts,
nothing grounds and you get `Query [score] not matched anywhere in the template`.

In [5]:
def build_model(layers=LAYERS, dim=DIM, act=Transformation.LEAKY_RELU):
    m = Model()

    # learnable embedding tables, as weighted facts
    for i in range(N_POS): m += R.emb_pos(f"p{i}")[dim,]
    for i in range(N_COL): m += R.emb_col(f"c{i}")[dim,]
    for i in range(N_SHP): m += R.emb_shp(f"s{i}")[dim,]
    for i in range(N_SIZ): m += R.emb_siz(f"z{i}")[dim,]
    for i in range(N_PNL): m += R.emb_pnl(f"g{i}")[dim,]

    # NodeEncoder: h0(O) = sum of the object's attribute embeddings
    for fact, emb in [("pos", "emb_pos"), ("color", "emb_col"), ("shape", "emb_shp"),
                      ("size", "emb_siz"), ("panel", "emb_pnl")]:
        m += (R.h0(V.O) <= (R.get(fact)(V.O, V.A), R.get(emb)(V.A)))
    m += (R.h0 / 1) | Metadata(transformation=Transformation.IDENTITY)

    # RelGNLayer x layers
    prev = "h0"
    for k in range(1, layers + 1):
        cur = f"h{k}"
        for rel in RELATIONS:
            m += (R.get(cur)(V.Dd) <= (
                      R.get(rel)(V.Ss, V.Dd),
                      R.get(prev)(V.Ss)[dim, dim],   # source state
                      R.get(prev)(V.Dd)[dim, dim],   # destination state — keeps node identity
                  )) | Metadata(aggregation=Aggregation.AVG)
        m += (R.get(cur)(V.X) <= R.get(prev)(V.X)[dim, dim]) | Metadata(aggregation=Aggregation.AVG)
        m += (R.get(cur) / 1) | Metadata(transformation=act)
        prev = cur

    # readout
    m += (R.ctx(V.G)  <= (R.graph(V.G), R.is_context(V.O),   R.get(prev)(V.O)[dim, dim])) \
         | Metadata(aggregation=Aggregation.AVG)
    m += (R.cand(V.G) <= (R.graph(V.G), R.is_candidate(V.O), R.get(prev)(V.O)[dim, dim])) \
         | Metadata(aggregation=Aggregation.AVG)
    m += (R.ctx / 1)  | Metadata(transformation=act)
    m += (R.cand / 1) | Metadata(transformation=act)
    # CONCAT, not the default SUM: torch feeds cat([ctx, cand]) to an MLP, and a plain sum
    # gives no ctx x cand interaction term at all.
    m += (R.score(V.G)[1, 2 * dim] <= (R.ctx(V.G)[dim, dim], R.cand(V.G)[dim, dim])) \
         | Metadata(combination=Combination.CONCAT)
    return m

## 5. Train

Each puzzle is 8 binary samples (1 correct, 7 wrong); at eval we `argmax` the 8 scores, so
chance is 12.5%.

`build_dataset()` can only be called **once** per built model in 0.9.2 — a second call throws
`ArrayIndexOutOfBoundsException`. So train and val are built together and the list is split.

In [6]:
print("loading…")
train_s, train_g = load("train", N_TRAIN)
val_s,   val_g   = load("val",   N_VAL)
print(f"{len(train_g)} train puzzles ({len(train_s)} samples), {len(val_g)} val puzzles")

model = build_model()
model.build(Settings(optimizer=Adam(lr=LR), error_function=CrossEntropy()))

all_built = list(model.build_dataset(Dataset(train_s + val_s)))   # single call!
tr, va = all_built[:len(train_s)], all_built[len(train_s):]
assert len(tr) == len(train_s) and len(va) == len(val_s), "samples were merged — check query ids"


def accuracy(built, groups):
    out = [float(np.asarray(y).flatten()[0]) for y in model.test(built)]
    hits = sum(int(np.argmax([out[i] for i in idx])) == t for idx, t in groups)
    return hits / len(groups), float(np.std(out))


print(f"\n{'epoch':>5} {'loss':>10} {'val acc':>9} {'out std':>9}    (chance 12.5%)")
best = 0.0
for ep in range(1, EPOCHS + 1):
    res = model.train(tr, epochs=1)
    loss = np.mean([float(np.asarray(r[2]).flatten()[0]) for r in res]) if res else float("nan")
    acc, spread = accuracy(va, val_g)
    best = max(best, acc)
    print(f"{ep:>5} {loss:>10.4f} {acc*100:>8.1f}% {spread:>9.2e}")
print(f"\nbest val acc: {best*100:.1f}%   (torch RelGNLayer on center_single: 75.4%)")

loading…


200 train puzzles (1600 samples), 80 val puzzles


+2839ms - 23:21:43:394 <cz.cvut.fel.ida.pipelines.Pipeline> (execute) [INFO] - Executing pipeline : PythonGroundingBuilding
+2856ms - 23:21:43:411 <cz.cvut.fel.ida.utils.generic.Utilities> (logGCStats) [WARNING] - Garbage collection takes 600% of time!!
+23ms - 23:21:43:417 <cz.cvut.fel.ida.pipelines.Pipeline> (execute) [INFO] - Executing pipeline : PythonGroundingbBuilding
+24ms - 23:21:43:441 <cz.cvut.fel.ida.pipelines.pipes.specific.SampleProgressLog> (sample) [INFO] - Grounding of sample 1: s_0: 0=score(train0_0) <- ex:#0 [facts:127]
+11ms - 23:21:43:452 <cz.cvut.fel.ida.pipelines.pipes.specific.SampleProgressLog> (sample) [INFO] - Neuralization of sample 1: s_0:score(train0_0): 0=WeightedAtomNeuron = score(train0_0) <- net:g0, neurons: 484
+22ms - 23:21:43:474 <cz.cvut.fel.ida.pipelines.pipes.specific.SampleProgressLog> (sample) [INFO] - Grounding of sample 2: s_1: 0=score(train0_1) <- ex:#1 [facts:127]
+3ms - 23:21:43:477 <cz.cvut.fel.ida.pipelines.pipes.specific.SampleProgressLo

+7ms - 23:21:43:540 <cz.cvut.fel.ida.pipelines.pipes.specific.SampleProgressLog> (sample) [INFO] - Grounding of sample 10: s_9: 0=score(train1_1) <- ex:#9 [facts:127]
+113ms - 23:21:43:653 <cz.cvut.fel.ida.pipelines.pipes.specific.SampleProgressLog> (sample) [INFO] - Grounding: further samples are summarised every 1000 rather than listed (loggedSampleDetails)
+2ms - 23:21:43:655 <cz.cvut.fel.ida.pipelines.pipes.specific.SampleProgressLog> (sample) [INFO] - Neuralization of sample 10: s_9:score(train1_1): 0=WeightedAtomNeuron = score(train1_1) <- net:g9, neurons: 482
+0ms - 23:21:43:655 <cz.cvut.fel.ida.pipelines.pipes.specific.SampleProgressLog> (sample) [INFO] - Neuralization: further samples are summarised every 1000 rather than listed (loggedSampleDetails)


+2494ms - 23:21:46:149 <cz.cvut.fel.ida.pipelines.pipes.specific.SampleProgressLog> (sample) [INFO] - Grounding: 1000 samples, head literals 30000 (30.0/sample), facts 127000 (127.0/sample), 2.8s
+1ms - 23:21:46:150 <cz.cvut.fel.ida.pipelines.pipes.specific.SampleProgressLog> (sample) [INFO] - Neuralization: 1000 samples, neurons 479856 (479.9/sample), 2.7s


...

   16     0.3800     20.0%  1.91e-01


   17     0.3805     21.2%  2.68e-01


   18     0.3800     20.0%  6.40e-02


   19     0.3872     21.2%  4.75e-02


   20     0.3797     15.0%  3.39e-02

best val acc: 21.2%   (torch RelGNLayer on center_single: 75.4%)


## 6. Why the accuracy is low

Four candidate explanations, tested one at a time. Two are real, two are not.

### It is not class imbalance
1 positive per 8 makes "always say no" tempting, and the plateau loss of **0.3810** is almost exactly
the **0.3768** a constant base-rate predictor scores — so the model really had learned nothing but the
base rate. But retraining on a *balanced* set (the positive plus one random negative per puzzle, 50/50)
left pairwise accuracy at 49-53% across 12 epochs — dead chance. Imbalance is not the blocker.

### It is not the loss shape alone
The torch model uses `F.cross_entropy(logits, target)` over `logits[batch, 8]` — one 8-way softmax
*ranking* the candidates ([gnn/train.py:102](gnn/train.py:102)). This port scores each candidate as an
independent binary, which is a weaker objective. Worth fixing eventually, but the balanced test above
rules it out as the main cause.

### Real problem 1: the readout had no interaction term
Body atoms combine with `Combination.SUM` by default, so `score = W1*ctx + W2*cand` — purely additive,
with no `ctx x cand` term. Since `ctx` is identical across a puzzle's 8 candidates, it contributes
nothing to the `argmax`. Torch instead feeds `cat([ctx, cand])` through an MLP. Measured, 12 epochs:

| readout | best val acc |
|---|---|
| `SUM` (default) | 13.8% |
| `ELPRODUCT` | 18.8% |
| `CONCAT` (torch-like) | 20.0% |

`CONCAT` is now the default above.

### Real problem 2: no LayerNorm, so activations saturate
Each layer sums 8 terms (7 relations + the residual — rules sharing a head combine with
`relation_combination = SUM`), and with `tanh` on top that saturates: every node maps to the same
vector and the eight candidate scores become identical. The `out std` column showed this directly,
decaying to ~1e-8 within a few epochs. Torch has `LayerNorm` inside every `RelGNLayer`; PyNeuraLogic
has no equivalent. Swapping the activation fixes the collapse:

| activation | out std, epoch 1 -> 12 | best val acc |
|---|---|---|
| `TANH` | 2.0e-04 -> **1.7e-08** (collapsed) | 16.2% |
| `NORM` | 2.5e-02 -> 1.4e-01 (stable) | 17.5% |
| `LEAKY_RELU` | 5.9e-02 -> 1.6e-01 (stable) | 18.8% |

`LEAKY_RELU` is now the default. Note the collapse is cured but accuracy barely moves — a healthy
output spread is necessary, not sufficient.

### And the honest one: this is a toy
Against the run that scored 75.4%, this notebook is roughly **1/24 the width, 1/30 the data, 1/30
the epochs**:

| | this notebook | torch best (I-RAVEN) |
|---|---|---|
| hidden dim | 16 | 256 embed / 384 hidden |
| layers | 2 | 3 |
| train puzzles | 200 (one constellation) | 6000 (all seven) |
| epochs | 20 | ~400 |

So low accuracy here is expected even with a correct port. The two defects above are worth fixing
because they would have capped a full-scale run too — but nothing in these results shows whether a
scaled-up version reaches parity. That takes a real training run on the cluster, via
[submit_pynl.sh](submit_pynl.sh).

**Next steps, in order:** (1) scale `DIM` to 128+, `N_TRAIN` to a few thousand, epochs to 200+, and
submit as a batch job rather than running in the notebook; (2) if it still stalls, replace the
per-candidate binary objective with a genuine 8-way ranking loss; (3) only then add `granular`
and the remaining six constellations.

Reference numbers, from the torch checkpoints in `experiments/`: `center_single` 75.4%, I-RAVEN
overall 68.8%. Ignore the 96.4% RAVEN figure — that benchmark's candidate set leaks the answer.